In [22]:
import pandas as pd

# Read Excel file

file_path = './data/sampling_logical_fallacy.xlsx.xlsx' #change the file name
df = None

print(f"Trying to read Excel file: {file_path}")

try:
    df = pd.read_excel(file_path)

    print(f"Success! Excel file '{file_path}' has been loaded.")

except FileNotFoundError:
    print(f"❌ Error: File not found -> {file_path}")

except ImportError:
    print("❌ Error: Missing required library!")
    print("Please install 'openpyxl' first: pip install openpyxl")

except Exception as e:
    print(f"❌ Other error occurred while reading Excel: {e}")


if df is not None:
    print("\nDataFrame info:")
    df.info()


else:
    print("\n❌ Failed to read file. Please check file path and library installation.")

Trying to read Excel file: final_result_clean.xlsx
Success! Excel file 'final_result_clean.xlsx' has been loaded.

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1905 entries, 0 to 1904
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   model_name            1905 non-null   object
 1   question_id           1905 non-null   int64 
 2   context               1905 non-null   object
 3   ground_truth          1898 non-null   object
 4   fallacy_option1       1786 non-null   object
 5   verification_status1  1786 non-null   object
 6   fallacy_option2       1554 non-null   object
 7   verification_status2  1554 non-null   object
 8   fallacy_option3       811 non-null    object
 9   verification_status3  811 non-null    object
 10  lean4_code1           1605 non-null   object
 11  explanation1          1786 non-null   object
 12  lean4_code2           1408 non-null   object
 13  explana

In [ ]:
# Detect cases where questions are labeled as no_fallacy
import pandas as pd
import numpy as np

# 1. Define option-related columns to check
# Based on DataFrame structure, from 'fallacy_option1' to 'explanation3'
option_columns = [
    'fallacy_option1', 'verification_status1', 'fallacy_option2', 'verification_status2',
    'fallacy_option3', 'verification_status3', 'lean4_code1', 'explanation1',
    'lean4_code2', 'explanation2', 'lean4_code3', 'explanation3'
]

# 2. Filter rows that match "Empty Case 1"

# Condition a: all option-related columns are empty (NaN / None)
# .isna() checks for empty values, .all(axis=1) ensures all columns are empty
condition_A = df[option_columns].isna().all(axis=1)
df_empty_case1 = df[condition_A].copy()


# 3. Group results by model_name

# Count how many Empty Case 1 questions per model
empty_case1_counts = (
    df_empty_case1
    .groupby('model_name')
    .size()
    .reset_index(name='Empty_Case1_Count')
)

# Get question_id list for each model
empty_case1_qids = (
    df_empty_case1
    .groupby('model_name')['question_id']
    .apply(list)
    .reset_index(name='Empty_Case1_QIDs')
)

# Merge results
df_diagnostic = pd.merge(
    empty_case1_counts,
    empty_case1_qids,
    on='model_name',
    how='outer'
)

# Format output so models without this case show 0
df_diagnostic = df_diagnostic.fillna(0)
df_diagnostic['Empty_Case1_Count'] = df_diagnostic['Empty_Case1_Count'].astype(int)


# 4. Print report
print("--- Model Diagnostic Report: Empty Case 1 (Context/GT present, but all options empty) ---")
print(df_diagnostic)

--- Model Diagnostic Report: Empty Case 1 (Context/GT present, but all options empty) ---
                            model_name  Empty_Case1_Count  \
0          anthropic/claude-sonnet-4.5                  7   
1          deepseek/deepseek-chat-v3.1                 14   
2                 deepseek/deepseek-r1                 14   
3                     gemini-flash-non                 10   
4                gemini-flash-thinking                  1   
5                       gemini-pro-non                  1   
6                  gemini-pro-thinking                  3   
7          moonshotai/kimi-k2-thinking                 14   
8             openai/gpt-4o-2024-11-20                  7   
9                         openai/gpt-5                  5   
10                 openai/gpt-oss-120b                 12   
11  qwen/qwen3-235b-a22b-thinking-2507                  8   
12           qwen_qwen3-235b-a22b-2507                  4   
13           x-ai/grok-4-fast-thinking                 1

In [16]:
#  Define constants
PASS_STATUS = 'lean_pass'

# Logical failure (new core error type)
LOGICAL_FAIL_STATUS = 'lean_pass_with_type_error'

# Technical failure (independent metric)
TECHNICAL_FAIL_STATUS = 'no_pass'


# Total rows per model (denominator)
total_rows = (
    df.groupby('model_name')
    .size()
    .reset_index(name='Total_Count')
)

In [ ]:
import pandas as pd
import numpy as np

# --- 0. Data cleaning and preparation ---

# Assume df is already loaded as a DataFrame

# 1. Remove rows where context is empty AND no output exists
# (these rows should not be counted in the denominator)
condition_to_delete = df['context'].isna() & df['ground_truth'].isna()

# Keep remaining rows
df_cleaned = df[~condition_to_delete].copy()

print(f"Original number of rows: {len(df)}")
print(f"Remaining rows after removing empty context rows: {len(df_cleaned)}")


# 2. Define option-related columns (used for no_fallacy@3 calculation)
option_columns = [
    'fallacy_option1', 'verification_status1',
    'fallacy_option2', 'verification_status2',
    'fallacy_option3', 'verification_status3',
    'lean4_code1', 'explanation1',
    'lean4_code2', 'explanation2',
    'lean4_code3', 'explanation3'
]

# Use cleaned data for all following calculations
df = df_cleaned

Original number of rows: 1905
Remaining rows after removing empty context rows: 1905


In [18]:
# Valid-Correct @3 (Label Match + Lean4 Pass)
valid_1 = (df['ground_truth'].eq(df['fallacy_option1']) & (df['verification_status1'] == PASS_STATUS)).fillna(False)
valid_2 = (df['ground_truth'].eq(df['fallacy_option2']) & (df['verification_status2'] == PASS_STATUS)).fillna(False)
valid_3 = (df['ground_truth'].eq(df['fallacy_option3']) & (df['verification_status3'] == PASS_STATUS)).fillna(False)
df['is_validated'] = valid_1 | valid_2 | valid_3

validated_df = df.groupby('model_name').agg(
    Validated_Count=('is_validated', 'sum')
).reset_index()
validated_df = pd.merge(validated_df, total_rows, on='model_name')
validated_df['Validated_Rate'] = validated_df['Validated_Count'] / validated_df['Total_Count']
validated_df['Validated_Rate'] = validated_df['Validated_Rate'].map('{:.2%}'.format)

print("\n--- A. Valid-Correct (Label Match + Lean4 Pass) ---")
print(validated_df)
print("-" * 40)


--- A. Valid-Correct (Label Match + Lean4 Pass) ---
                            model_name  Validated_Count  Total_Count  \
0          anthropic/claude-sonnet-4.5               45          127   
1          deepseek/deepseek-chat-v3.1               35          127   
2                 deepseek/deepseek-r1               37          127   
3                     gemini-flash-non               40          127   
4                gemini-flash-thinking               45          127   
5                       gemini-pro-non               47          127   
6                  gemini-pro-thinking               54          127   
7          moonshotai/kimi-k2-thinking               44          127   
8             openai/gpt-4o-2024-11-20               49          127   
9                         openai/gpt-5               52          127   
10                 openai/gpt-oss-120b               42          127   
11  qwen/qwen3-235b-a22b-thinking-2507               45          127   
12         

In [19]:
# Valid-Alternative @3 (Label Mismatch + Lean 4 Pass)
pass_1_base = (df['fallacy_option1'].notna() & (df['verification_status1'] == PASS_STATUS)).fillna(False)
pass_2_base = (df['fallacy_option2'].notna() & (df['verification_status2'] == PASS_STATUS)).fillna(False)
pass_3_base = (df['fallacy_option3'].notna() & (df['verification_status3'] == PASS_STATUS)).fillna(False)
df['has_any_pass'] = pass_1_base | pass_2_base | pass_3_base

df['is_alt'] = df['has_any_pass'] & (~df['is_validated'])

alt_df = df.groupby('model_name').agg(
    Alt_Count=('is_alt', 'sum')
).reset_index()
alt_df = pd.merge(alt_df, total_rows, on='model_name')
alt_df['Alt_Rate'] = alt_df['Alt_Count'] / alt_df['Total_Count']
alt_df['Alt_Rate'] = alt_df['Alt_Rate'].map('{:.2%}'.format)

print("\n--- B. Valid-Alternative (Label Mismatch + Lean 4 Pass) ---")
print(alt_df)
print("-" * 40)


--- B. Valid-Alternative (Label Mismatch + Lean 4 Pass) ---
                            model_name  Alt_Count  Total_Count Alt_Rate
0          anthropic/claude-sonnet-4.5         73          127   57.48%
1          deepseek/deepseek-chat-v3.1         75          127   59.06%
2                 deepseek/deepseek-r1         76          127   59.84%
3                     gemini-flash-non         76          127   59.84%
4                gemini-flash-thinking         73          127   57.48%
5                       gemini-pro-non         77          127   60.63%
6                  gemini-pro-thinking         70          127   55.12%
7          moonshotai/kimi-k2-thinking         69          127   54.33%
8             openai/gpt-4o-2024-11-20         71          127   55.91%
9                         openai/gpt-5         70          127   55.12%
10                 openai/gpt-oss-120b         73          127   57.48%
11  qwen/qwen3-235b-a22b-thinking-2507         72          127   56.69%
12 

In [20]:
# C. Invalid-Correct @3 (Label Match + Lean 4 Fail)

def is_lucky_base_new(option_col, status_col):
    return (
        df[option_col].notna()
        & df[option_col].eq(df['ground_truth'])
        & (df[status_col] == LOGICAL_FAIL_STATUS) # 僅檢查 pass_with_error
    ).fillna(False)

lucky_base_1 = is_lucky_base_new('fallacy_option1', 'verification_status1')
lucky_base_2 = is_lucky_base_new('fallacy_option2', 'verification_status2')
lucky_base_3 = is_lucky_base_new('fallacy_option3', 'verification_status3')
df['has_lucky_guess_base'] = lucky_base_1 | lucky_base_2 | lucky_base_3
is_already_classified_1_2 = df['is_validated'] | df['is_alt'] 

df['is_lucky'] = df['has_lucky_guess_base'] & (~is_already_classified_1_2)

lucky_df = df.groupby('model_name').agg(Lucky_Count=('is_lucky', 'sum')).reset_index()
lucky_df = pd.merge(lucky_df, total_rows, on='model_name')
lucky_df['Lucky_Rate'] = lucky_df['Lucky_Count'] / lucky_df['Total_Count']
lucky_df['Lucky_Rate'] = lucky_df['Lucky_Rate'].map('{:.2%}'.format)

print("\n--- C. Invalid-Correct (Label Match + Lean 4 Fail) ---")
print(lucky_df)
print("-" * 40)


--- C. Invalid-Correct (Label Match + Lean 4 Fail) ---
                            model_name  Lucky_Count  Total_Count Lucky_Rate
0          anthropic/claude-sonnet-4.5            1          127      0.79%
1          deepseek/deepseek-chat-v3.1            0          127      0.00%
2                 deepseek/deepseek-r1            0          127      0.00%
3                     gemini-flash-non            0          127      0.00%
4                gemini-flash-thinking            2          127      1.57%
5                       gemini-pro-non            1          127      0.79%
6                  gemini-pro-thinking            0          127      0.00%
7          moonshotai/kimi-k2-thinking            0          127      0.00%
8             openai/gpt-4o-2024-11-20            0          127      0.00%
9                         openai/gpt-5            0          127      0.00%
10                 openai/gpt-oss-120b            0          127      0.00%
11  qwen/qwen3-235b-a22b-thinkin

In [21]:
# Invalid-Incorrect @3 (Label Mismatch + Lean 4 Fail)
def is_invalid_base(option_col, status_col):
    return (
        df[option_col].notna()
        & (df[status_col] == LOGICAL_FAIL_STATUS) # 僅檢查 pass_with_error
    ).fillna(False)

invalid_base_1 = is_invalid_base('fallacy_option1', 'verification_status1')
invalid_base_2 = is_invalid_base('fallacy_option2', 'verification_status2')
invalid_base_3 = is_invalid_base('fallacy_option3', 'verification_status3')
df['is_invalid_base'] = invalid_base_1 | invalid_base_2 | invalid_base_3

# Invalid 不應用互斥邏輯，計算所有 "Any Fail"
# 2. 應用互斥邏輯
is_already_classified_1_3 = df['is_validated'] | df['is_alt'] | df['is_lucky']
df['is_invalid'] = df['is_invalid_base'] & (~is_already_classified_1_3)

invalid_df = df.groupby('model_name').agg(
    Invalid_Count=('is_invalid', 'sum')
).reset_index()

invalid_df = pd.merge(invalid_df, total_rows, on='model_name')
invalid_df['Invalid_Rate'] = invalid_df['Invalid_Count'] / invalid_df['Total_Count']
invalid_df['Invalid_Rate'] = invalid_df['Invalid_Rate'].map('{:.2%}'.format)

print("\n--- D. Invalid-Incorrect (Label Mismatch + Lean 4 Fail) ---")
print(invalid_df)
print("-" * 40)



--- D. Invalid-Incorrect (Label Mismatch + Lean 4 Fail) ---
                            model_name  Invalid_Count  Total_Count  \
0          anthropic/claude-sonnet-4.5              1          127   
1          deepseek/deepseek-chat-v3.1              2          127   
2                 deepseek/deepseek-r1              0          127   
3                     gemini-flash-non              0          127   
4                gemini-flash-thinking              5          127   
5                       gemini-pro-non              1          127   
6                  gemini-pro-thinking              0          127   
7          moonshotai/kimi-k2-thinking              0          127   
8             openai/gpt-4o-2024-11-20              0          127   
9                         openai/gpt-5              0          127   
10                 openai/gpt-oss-120b              0          127   
11  qwen/qwen3-235b-a22b-thinking-2507              1          127   
12           qwen_qwen3-235b-

In [22]:
# no_fallacy @3

# --- 1. Calculate the final corrected No_Fallacy@3
# This definition covers all cases where option-related fields are missing ---
is_already_classified_1_4 = (
    df['is_validated'] |
    df['is_alt'] |
    df['is_lucky'] |
    df['is_invalid']
)

# Condition: all option-related columns are empty
# (ground_truth may be empty or non-empty)
condition_no_fallacy_final = df[option_columns].isna().all(axis=1)

# Mark rows where all option-related columns are empty
df['is_no_fallacy_final'] = condition_no_fallacy_final & (~is_already_classified_1_4)

# Calculate results
# (simple counting; mutual exclusiveness has not been applied yet)
no_fallacy_df = df.groupby('model_name').agg(
    No_Fallacy_Count=('is_no_fallacy_final', 'sum'),
    Total_Count=('question_id', 'count')
).reset_index()

no_fallacy_df['No_Fallacy_Rate'] = no_fallacy_df['No_Fallacy_Count'] / no_fallacy_df['Total_Count']
no_fallacy_df['No_Fallacy_Rate'] = no_fallacy_df['No_Fallacy_Rate'].map('{:.2%}'.format)

print("--- 1. No_Fallacy@3 (Final definition: Context is present, while all option and verification fields are empty) ---")
print(no_fallacy_df)
print("-" * 40)

--- 1. No_Fallacy@3 (Final definition: Context is present, while all option and verification fields are empty) ---
                            model_name  No_Fallacy_Count  Total_Count  \
0          anthropic/claude-sonnet-4.5                 7          127   
1          deepseek/deepseek-chat-v3.1                14          127   
2                 deepseek/deepseek-r1                14          127   
3                     gemini-flash-non                10          127   
4                gemini-flash-thinking                 1          127   
5                       gemini-pro-non                 1          127   
6                  gemini-pro-thinking                 3          127   
7          moonshotai/kimi-k2-thinking                14          127   
8             openai/gpt-4o-2024-11-20                 7          127   
9                         openai/gpt-5                 5          127   
10                 openai/gpt-oss-120b                12          127   
11  qwen/

In [23]:
# E. Syntax Failure @3

def is_technical_fail_base(option_col, status_col):
    return (
        df[option_col].notna()
        & (df[status_col] == TECHNICAL_FAIL_STATUS) # only no_pass
    ).fillna(False)

tech_fail_base_1 = is_technical_fail_base('fallacy_option1', 'verification_status1')
tech_fail_base_2 = is_technical_fail_base('fallacy_option2', 'verification_status2')
tech_fail_base_3 = is_technical_fail_base('fallacy_option3', 'verification_status3')
df['is_technical_fail_base'] = tech_fail_base_1 | tech_fail_base_2 | tech_fail_base_3
is_already_classified_1_5 = df['is_validated'] | df['is_alt'] | df['is_lucky'] | df['is_invalid'] | df['is_no_fallacy_final']

df['is_technical_fail'] = df['is_technical_fail_base']  & (~is_already_classified_1_5)

tech_fail_df = df.groupby('model_name').agg(Tech_Fail_Count=('is_technical_fail', 'sum')).reset_index()
tech_fail_df = pd.merge(tech_fail_df, total_rows, on='model_name')
tech_fail_df['Tech_Fail_Rate'] = tech_fail_df['Tech_Fail_Count'] / tech_fail_df['Total_Count']
tech_fail_df['Tech_Fail_Rate'] = tech_fail_df['Tech_Fail_Rate'].map('{:.2%}'.format)

print("\n--- E. Syntax Failure ---")
print(tech_fail_df)
print("-" * 40)


--- E. Syntax Failure ---
                            model_name  Tech_Fail_Count  Total_Count  \
0          anthropic/claude-sonnet-4.5                0          127   
1          deepseek/deepseek-chat-v3.1                1          127   
2                 deepseek/deepseek-r1                0          127   
3                     gemini-flash-non                1          127   
4                gemini-flash-thinking                1          127   
5                       gemini-pro-non                0          127   
6                  gemini-pro-thinking                0          127   
7          moonshotai/kimi-k2-thinking                0          127   
8             openai/gpt-4o-2024-11-20                0          127   
9                         openai/gpt-5                0          127   
10                 openai/gpt-oss-120b                0          127   
11  qwen/qwen3-235b-a22b-thinking-2507                1          127   
12           qwen_qwen3-235b-a22b-250